# Evaluate Baseline vs Fine-tuned on the CORD-v2 **Validation** set

Uses the same engine as the app (`receipt_app_multi.py`: 4-bit base model + LoRA adapter on/off, run through `AssembledVLM`) and the same metrics as the assembly notebook, so numbers are comparable.

- **Baseline** = pretrained Qwen2.5-VL-3B (downloaded from Hugging Face, adapter disabled)
- **Fine-tuned** = baseline + your LoRA adapter folder (`lora_receipt_3b_trl`)

**Before running:** Runtime > Change runtime type > **T4 GPU**. Have `receipt_app_multi.py` and `lora_receipt_3b_trl.zip` ready to upload.

## 1. Setup

In [ ]:
%%capture
!pip install -q -U transformers peft bitsandbytes accelerate datasets opencv-python-headless pandas

## 2. Upload files
Uploads `receipt_app_multi.py` and the adapter zip (skipped if already present).

In [ ]:
import os, zipfile
from google.colab import files

ADAPTER_DIR = 'lora_receipt_3b_trl'

need = []
if not os.path.isfile('receipt_app_multi.py'): need.append('receipt_app_multi.py')
if not os.path.isdir(ADAPTER_DIR): need.append('lora_receipt_3b_trl.zip')
if need:
    print('Upload:', need)
    for name in files.upload():
        if name.endswith('.zip'):
            with zipfile.ZipFile(name) as z: z.extractall('.')

print('app file :', os.path.isfile('receipt_app_multi.py'))
print('adapter  :', sorted(os.listdir(ADAPTER_DIR))[:6] if os.path.isdir(ADAPTER_DIR) else 'NOT FOUND')

## 3. Configuration
`N_VAL = None` evaluates all 100 validation receipts. Use a small number (e.g. 5) for a quick smoke test.

In [ ]:
N_VAL = None          # None = whole validation split
SPLIT = 'validation'

## 4. Load models and data

In [ ]:
import json, time
import pandas as pd
from tqdm.auto import tqdm
from datasets import load_dataset
import receipt_app_multi as app

engine = app.Engine({'finetuned': ADAPTER_DIR})      # baseline + finetuned share one 4-bit base model
print('modes:', engine.modes)

val = load_dataset('naver-clova-ix/cord-v2', split=SPLIT)
if N_VAL: val = val.select(range(N_VAL))
print(SPLIT, 'receipts:', len(val))

## 5. Metric helpers
Same definitions as the assembly notebook (`receipt_accuracy`, `get_total`).

In [ ]:
def get_total(d):
    t = d.get('total') if isinstance(d, dict) else None
    if isinstance(t, dict): v = t.get('total_price')
    elif isinstance(t, list) and t and isinstance(t[0], dict): v = t[0].get('total_price')
    else: v = None
    return str(v) if v is not None else None

## 6. Run the evaluation
Each model is run over every validation receipt. Raw outputs are kept so you can inspect errors afterwards.

In [ ]:
import torch
records = []
summary = []
for mode in engine.modes:
    torch.cuda.reset_peak_memory_stats()
    accs, times = [], []
    total_ok = total_n = parse_fail = 0
    for i in tqdm(range(len(val)), desc=mode):
        row = val[i]
        gt = json.loads(row['ground_truth'])['gt_parse']
        pred, raw, secs = engine.extract(row['image'], mode)
        acc = app.field_accuracy(gt, pred)
        g = get_total(gt)
        tot_ok = None
        if g is not None:
            total_n += 1
            tot_ok = bool(pred is not None and get_total(app.normalize(pred)) == g)
            total_ok += int(tot_ok)
        parse_fail += int(pred is None)
        accs.append(acc); times.append(secs)
        records.append({'idx': i, 'mode': mode, 'field_accuracy': acc, 'total_match': tot_ok,
                        'valid_json': pred is not None, 'seconds': secs,
                        'ground_truth': json.dumps(gt), 'raw_output': raw})
    summary.append({'model': app.MODE_LABELS[mode], 'n_receipts': len(val),
                    'field_accuracy_%': round(100 * sum(accs) / len(accs), 2),
                    'total_price_match_%': round(100 * total_ok / max(total_n, 1), 2),
                    'json_parse_failures': parse_fail,
                    'sec_per_receipt': round(sum(times) / len(times), 2),
                    'peak_vram_gb': round(torch.cuda.max_memory_allocated() / 1e9, 2)})

results = pd.DataFrame(summary)
per_receipt = pd.DataFrame(records)
results

## 7. Save results

In [ ]:
results.to_csv(f'results_{SPLIT}.csv', index=False)
per_receipt.to_csv(f'per_receipt_{SPLIT}.csv', index=False)
print(results.to_string(index=False))

from google.colab import files
files.download(f'results_{SPLIT}.csv')
files.download(f'per_receipt_{SPLIT}.csv')

## 8. Where does the fine-tuned model win or lose?
Per-receipt comparison (positive delta = fine-tuned better).

In [ ]:
piv = per_receipt.pivot(index='idx', columns='mode', values='field_accuracy')
if 'finetuned' in piv:
    piv['delta'] = piv['finetuned'] - piv['baseline']
    print('fine-tuned better:', int((piv.delta > 0).sum()), '| worse:', int((piv.delta < 0).sum()), '| equal:', int((piv.delta == 0).sum()))
    display(piv.sort_values('delta').head(5)); display(piv.sort_values('delta').tail(5))